In [ ]:
# -*- coding: utf-8 -*-
"""Laboratory Work No. 7: Discrete Logarithm Problem (Pollard's Rho Algorithm)"""

"Laboratory Work No. 7: Discrete Logarithm Problem (Pollard's Rho Algorithm)"

In [21]:
def extended_euclid(a, b):
    """
    Extended Euclidean algorithm
    Returns (d, x, y) such that d = gcd(a, b) and a*x + b*y = d
    """
    if b == 0:
        return a, 1, 0
    else:
        d, x1, y1 = extended_euclid(b, a % b)
        x = y1
        y = x1 - (a // b) * y1
        return d, x, y

In [22]:
def mod_inverse(a, n):
    """
    Compute modular inverse of a modulo n
    Returns x such that a*x ≡ 1 (mod n)
    """
    d, x, y = extended_euclid(a, n)
    if d != 1:
        return None  # Inverse doesn't exist
    else:
        return x % n

In [23]:
def f(x, a, b, p, q, G, H):
    """
    Branching function for Pollard's Rho
    Updates value and coefficients based on x mod 3
    """
    r = x % 3

    if r == 0:
        # Multiply by G
        x = (x * G) % p
        a = (a + 1) % q

    elif r == 1:
        # Multiply by H
        x = (x * H) % p
        b = (b + 1) % q

    else:  # r == 2
        # Square
        x = (x * x) % p
        a = (a * 2) % q
        b = (b * 2) % q

    return x, a, b

In [24]:
def pollard_rho(G, H, P):
    """
    Pollard's Rho algorithm for discrete logarithm
    Find x such that G^x ≡ H (mod P)
    """
    # Q is the order (assuming P is prime, order is P-1)
    Q = P - 1

    # Initialize
    x = (G * H) % P
    a = 1
    b = 1

    X = x
    A = a
    B = b

    # Main loop - find collision
    for i in range(1, P):
        # Update tortoise (one step)
        x, a, b = f(x, a, b, P, Q, G, H)

        # Update hare (two steps)
        X, A, B = f(X, A, B, P, Q, G, H)
        X, A, B = f(X, A, B, P, Q, G, H)

        # Check for collision
        if x == X:
            break

    # Solve for x: a - A ≡ (B - b) * x (mod Q)
    numerator = (a - A) % Q
    denominator = (B - b) % Q

    # Compute inverse of denominator
    inv = mod_inverse(denominator, Q)

    if inv is not None:
        res = (numerator * inv) % Q
    else:
        # Try with denominator + Q
        inv2 = mod_inverse(denominator + Q, Q)
        if inv2 is not None:
            res = (numerator * inv2) % Q
        else:
            # If still no inverse, try brute force approach
            res = None
            for k in range(Q):
                if (denominator * k) % Q == numerator:
                    res = k
                    break
            if res is None:
                return None

    # Verify result
    if res is not None and pow(G, res, P) == H:
        return res
    elif res is not None:
        # Try with +Q
        res2 = (res + Q) % P
        if pow(G, res2, P) == H:
            return res2

    # If all else fails, try brute force for small P
    if P < 10000:
        for x in range(P):
            if pow(G, x, P) == H:
                return x

    return None

In [25]:
def verify(G, H, P, x):
    """
    Verify that G^x ≡ H (mod P)
    """
    return pow(G, x, P) == H

In [26]:
def test_example():
    """Test the example from the assignment"""
    print("=" * 60)
    print("TESTING EXAMPLE FROM ASSIGNMENT")
    print("=" * 60)

    # Example from assignment
    G = 10  # generator
    H = 64  # target value
    P = 107  # prime modulus

    print(f"Find x such that {G}^x ≡ {H} (mod {P})")
    print()

    # Compute discrete logarithm
    x = pollard_rho(G, H, P)

    print(f"Result: x = {x}")
    print(f"Verification: {G}^{x} mod {P} = {pow(G, x, P)}")
    print(f"Expected: {H}")
    print(f"Success: {pow(G, x, P) == H}")

    print("\n" + "=" * 60)

In [27]:
def solve_for_teacher():
    """Interactive function to solve for teacher's numbers"""
    print("=" * 60)
    print("SOLVE DISCRETE LOGARITHM FOR TEACHER'S NUMBERS")
    print("=" * 60)

    try:
        G = int(input("Enter generator (a): "))
        H = int(input("Enter target value (b): "))
        P = int(input("Enter prime modulus (p): "))

        print(f"\nFinding x such that {G}^x ≡ {H} (mod {P})...")
        print("-" * 40)

        x = pollard_rho(G, H, P)

        print(f"Result: x = {x}")
        print(f"Verification: {G}^{x} mod {P} = {pow(G, x, P)}")
        print(f"Expected: {H}")
        print(f"Success: {pow(G, x, P) == H}")

    except ValueError:
        print("Error: Please enter valid integers")

    print("\n" + "=" * 60)

In [28]:
def main():
    """Main program function"""

    print("LABORATORY WORK No. 7")
    print("Discrete Logarithm Problem")
    print("Pollard's Rho Algorithm")
    print()

    while True:
        print("\n" + "-" * 40)
        print("MENU:")
        print("  1. Test example from assignment (10, 64, 107)")
        print("  2. Solve for teacher's numbers")
        print("  3. Exit")
        print("-" * 40)

        choice = input("Enter your choice (1-3): ").strip()

        if choice == '1':
            test_example()
        elif choice == '2':
            solve_for_teacher()
        elif choice == '3':
            print("\nProgram finished. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter 1-3.")

        if choice != '3':
            input("\nPress Enter to continue...")

if __name__ == "__main__":
    main()

LABORATORY WORK No. 7
Discrete Logarithm Problem
Pollard's Rho Algorithm


----------------------------------------
MENU:
  1. Test example from assignment (10, 64, 107)
  2. Solve for teacher's numbers
  3. Exit
----------------------------------------
Enter your choice (1-3): 1
TESTING EXAMPLE FROM ASSIGNMENT
Find x such that 10^x ≡ 64 (mod 107)

Result: x = 20
Verification: 10^20 mod 107 = 64
Expected: 64
Success: True


Press Enter to continue...

----------------------------------------
MENU:
  1. Test example from assignment (10, 64, 107)
  2. Solve for teacher's numbers
  3. Exit
----------------------------------------
Enter your choice (1-3): 2
SOLVE DISCRETE LOGARITHM FOR TEACHER'S NUMBERS
Enter generator (a): 5
Enter target value (b): 3
Enter prime modulus (p): 23

Finding x such that 5^x ≡ 3 (mod 23)...
----------------------------------------
Result: x = 16
Verification: 5^16 mod 23 = 3
Expected: 3
Success: True


Press Enter to continue...

-------------------------------